# 01 — Train the 1D CNN

**Purpose:** train a CREB1 binding classifier on the dataset produced by `00_data_sanity.ipynb`.

This notebook is a **thin consumer** of `src/`. All training logic lives in `src/train.py`, so the same code can be run from the terminal (`python -m src.train`) or in this notebook. If you find yourself writing a training loop here, put it in `src/train.py` instead.

**Model:** `Creb1CNN` from `src/model.py` — a 3-block 1D CNN with adaptive average pooling, producing a single logit per sequence.

**Training config:**
- Loss: `BCEWithLogitsLoss` (sigmoid + BCE, numerically stable)
- Optimizer: Adam, lr = 1e-3
- Batch size: 64
- Epochs: 20
- Split: test chromosomes `1, 2`; validation is a random 10% of the training rows (this is for monitoring only — the leak-free split is train↔test).

**Outputs:** `checkpoints/cnn.pt` (best-val-loss checkpoint) and an in-memory loss history.

In [1]:
from src.notebook_setup import set_repo_root
set_repo_root()

In [2]:
%load_ext autoreload
%autoreload 2

import numpy as np
import torch
import matplotlib.pyplot as plt

from src.data import DataConfig, build_dataset, split_by_chromosome
from src.model import Creb1CNN
from src.train import make_loaders, train, save_checkpoint

Matplotlib is building the font cache; this may take a moment.


## 1. Configuration

All hyperparameters are plain variables here — edit and re-run the training cell without touching `src/`.

**Note on `test_chroms`:** chromosome-level holdout is the standard defense against near-duplicate leakage in genomics models. `("1", "2")` follows the convention used by Basenji, Enformer, and most TF-binding benchmarks.

**Note on the validation split:** validation is a *random row split* within the training chromosomes, not a chromosome holdout. This is intentional — val loss is for monitoring overfitting during training, not for reporting final numbers. The leak-free number is the test AUROC computed in `02_eval.ipynb`.